# Manual Organism Pipeline

This notebook lets you manually run the organism-creation pipeline with the repo's real prompt assets and evaluators.

Use it to:
- choose a config preset and scoring experiment
- build crossover prompts from pasted parent artifacts
- build mutation prompts from pasted parent artifacts
- build implementation prompts from pasted child artifacts
- run a simple evaluation for one `implementation.py`


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repo root or the notebooks/ directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import JSON, display

from src.evolve.manual_pipeline import (
    build_manual_crossover_prompts,
    build_manual_implementation_prompts,
    build_manual_mutation_prompts,
    load_manual_pipeline_context,
    run_manual_simple_evaluation,
)


def show_prompt_exchange(payload):
    metadata = {
        key: value
        for key, value in payload.items()
        if key not in {"system_prompt", "user_prompt"}
    }
    if metadata:
        display(JSON(metadata, expanded=False))
    print("=== SYSTEM PROMPT ===\n")
    print(payload["system_prompt"])
    print("\n=== USER PROMPT ===\n")
    print(payload["user_prompt"])


In [ ]:
CONFIG_NAME = "config_circle_packing_shinka"
EXPERIMENT_NAME = "unit_square_26"  # Required when the preset contains multiple experiments.
CONFIG_OVERRIDES = []
NOTEBOOK_SEED = 42

context = load_manual_pipeline_context(
    config_name=CONFIG_NAME,
    experiment_name=EXPERIMENT_NAME,
    overrides=CONFIG_OVERRIDES,
)

print(f"Config preset: {context.config_name}")
print(f"Scoring experiment: {context.experiment_name}")
print("Available experiments:")
for name in context.available_experiments:
    print(f"- {name}")


## 1. Crossover Design

Paste the mother's and father's `genetic_code.md` contents below. The run cell samples the inherited child gene pool with the same crossover probability used by the configured evolver, unless you override it.

You can leave `FATHER_LINEAGE_JSON = "[]"` if you do not want to provide paternal history.


In [ ]:
MOTHER_GENETIC_CODE = """## CORE_GENES
- Paste mother gene 1 here
- Paste mother gene 2 here
- Paste mother gene 3 here

## INTERACTION_NOTES
Paste mother interaction notes here.

## COMPUTE_NOTES
Paste mother compute notes here.
"""


In [ ]:
FATHER_GENETIC_CODE = """## CORE_GENES
- Paste father gene 1 here
- Paste father gene 2 here
- Paste father gene 3 here

## INTERACTION_NOTES
Paste father interaction notes here.

## COMPUTE_NOTES
Paste father compute notes here.
"""


In [ ]:
MOTHER_LINEAGE_JSON = "[]"
FATHER_LINEAGE_JSON = "[]"
CROSSOVER_INHERIT_PROBABILITY = None  # Use config default when None.
CROSSOVER_NOVELTY_FEEDBACK = []

crossover_payload = build_manual_crossover_prompts(
    context,
    mother_genetic_code_text=MOTHER_GENETIC_CODE,
    father_genetic_code_text=FATHER_GENETIC_CODE,
    mother_lineage_json=MOTHER_LINEAGE_JSON,
    father_lineage_json=FATHER_LINEAGE_JSON,
    inherit_probability=CROSSOVER_INHERIT_PROBABILITY,
    seed=NOTEBOOK_SEED,
    novelty_feedback=CROSSOVER_NOVELTY_FEEDBACK,
)

show_prompt_exchange(crossover_payload)


## 2. Mutation Design

Paste the parent `genetic_code.md` and optional `lineage.json`. The run cell prunes the parent gene pool with the configured mutation probability unless you override it.


In [ ]:
PARENT_GENETIC_CODE = """## CORE_GENES
- Paste parent gene 1 here
- Paste parent gene 2 here
- Paste parent gene 3 here

## INTERACTION_NOTES
Paste parent interaction notes here.

## COMPUTE_NOTES
Paste parent compute notes here.
"""


In [ ]:
PARENT_LINEAGE_JSON = "[]"
MUTATION_DELETE_PROBABILITY = None  # Use config default when None.
MUTATION_NOVELTY_FEEDBACK = []

mutation_payload = build_manual_mutation_prompts(
    context,
    parent_genetic_code_text=PARENT_GENETIC_CODE,
    parent_lineage_json=PARENT_LINEAGE_JSON,
    delete_probability=MUTATION_DELETE_PROBABILITY,
    seed=NOTEBOOK_SEED,
    novelty_feedback=MUTATION_NOVELTY_FEEDBACK,
)

show_prompt_exchange(mutation_payload)


## 3. Implementation Prompt

Paste the final child `genetic_code.md` and the child novelty summary (`CHANGE_DESCRIPTION` / organism novelty summary). The output is the system and user prompt pair for generating `implementation.py`.


In [ ]:
CHILD_GENETIC_CODE = """## CORE_GENES
- Paste final child gene 1 here
- Paste final child gene 2 here
- Paste final child gene 3 here

## INTERACTION_NOTES
Paste child interaction notes here.

## COMPUTE_NOTES
Paste child compute notes here.
"""

CHILD_NOVELTY_SUMMARY = "Paste the final novelty summary here."


In [ ]:
implementation_payload = build_manual_implementation_prompts(
    context,
    organism_genetic_code_text=CHILD_GENETIC_CODE,
    novelty_summary=CHILD_NOVELTY_SUMMARY,
)

show_prompt_exchange(implementation_payload)


## 4. Simple Score

Point this at the generated `implementation.py`. If the file is already named `implementation.py`, the evaluator uses its parent directory directly. Otherwise the helper copies it into a temporary alias directory under `.tmp_manual_pipeline/`.


In [ ]:
IMPLEMENTATION_PATH = "/absolute/path/to/implementation.py"
EVAL_MODE = "smoke"  # "smoke" or "full"
EVAL_DEVICE = "cpu"
EVAL_PRECISION = "fp32"

evaluation_report = run_manual_simple_evaluation(
    context,
    implementation_path=IMPLEMENTATION_PATH,
    mode=EVAL_MODE,
    device=EVAL_DEVICE,
    precision=EVAL_PRECISION,
    seed=NOTEBOOK_SEED,
)

display(JSON(evaluation_report, expanded=True))
print(f"\nscore = {evaluation_report['score']}")
